In [ ]:
!pip install torch-geometric -q
!pip install transformers -q

In [ ]:
import os
import random
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F

from collections import defaultdict

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool

from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
BASE_PATH = "/content/drive/MyDrive/CoAID/"

DATES = ["05-01-2020", "07-01-2020", "09-01-2020", "11-01-2020"]

all_news = []
all_tweets = []
all_replies = []

for d in DATES:
    path = os.path.join(BASE_PATH, d)

    fake_news = pd.read_csv(os.path.join(path, "NewsFakeCOVID-19.csv"))
    fake_news["label"] = 1

    real_news = pd.read_csv(os.path.join(path, "NewsRealCOVID-19.csv"))
    real_news["label"] = 0

    fake_tweets = pd.read_csv(os.path.join(path, "NewsFakeCOVID-19_tweets.csv"))
    real_tweets = pd.read_csv(os.path.join(path, "NewsRealCOVID-19_tweets.csv"))

    fake_replies = pd.read_csv(os.path.join(path, "NewsFakeCOVID-19_tweets_replies.csv"))
    real_replies = pd.read_csv(os.path.join(path, "NewsRealCOVID-19_tweets_replies.csv"))

    fake_news.rename(columns={"Unnamed: 0": "news_id"}, inplace=True)
    real_news.rename(columns={"Unnamed: 0": "news_id"}, inplace=True)
    fake_tweets.rename(columns={"index": "news_id"}, inplace=True)
    real_tweets.rename(columns={"index": "news_id"}, inplace=True)
    all_news.append(pd.concat([fake_news, real_news]))
    all_tweets.append(pd.concat([fake_tweets, real_tweets]))
    all_replies.append(pd.concat([fake_replies, real_replies]))

all_news = pd.concat(all_news).drop_duplicates("news_id")
all_tweets = pd.concat(all_tweets)
all_replies = pd.concat(all_replies)

print("Total news:", len(all_news))

Total news: 4532


In [ ]:
all_news["label"].value_counts()

,count
label,
0,3960
1,572


In [ ]:


tweet_counts = all_tweets.groupby("news_id").size()

fake_tweet_count = all_tweets[all_tweets["news_id"].isin(
    all_news[all_news["label"] == 1]["news_id"]
)].shape[0]

real_tweet_count = all_tweets[all_tweets["news_id"].isin(
    all_news[all_news["label"] == 0]["news_id"]
)].shape[0]

print("Fake tweet edges:", fake_tweet_count)
print("Real tweet edges:", real_tweet_count)

Fake tweet edges: 27265
Real tweet edges: 125023


In [ ]:
fake_reply_count = all_replies[all_replies["news_id"].isin(
    all_news[all_news["label"] == 1]["news_id"]
)].shape[0]

real_reply_count = all_replies[all_replies["news_id"].isin(
    all_news[all_news["label"] == 0]["news_id"]
)].shape[0]

print("Fake reply edges:", fake_reply_count)
print("Real reply edges:", real_reply_count)

Fake reply edges: 12807
Real reply edges: 109786


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   news_id         31 non-null     int64  
 1   type            31 non-null     object 
 2   fact_check_url  31 non-null     object 
 3   archive         26 non-null     object 
 4   news_url        31 non-null     object 
 5   news_url2       4 non-null      object 
 6   news_url3       3 non-null      object 
 7   news_url4       1 non-null      object 
 8   news_url5       0 non-null      float64
 9   title           31 non-null     object 
 10  newstitle       31 non-null     object 
 11  content         19 non-null     object 
 12  abstract        14 non-null     object 
 13  publish_date    7 non-null      object 
 14  meta_keywords   31 non-null     object 
 15  label           31 non-null     int64  
dtypes: float64(1), int64(2), object(13)
memory usage: 4.0+ KB
<class 'pandas.core.fram

In [ ]:
all_news.info()
all_tweets.info()
all_replies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4532 entries, 0 to 921
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   news_id         4532 non-null   int64 
 1   type            4506 non-null   object
 2   fact_check_url  4532 non-null   object
 3   archive         203 non-null    object
 4   news_url        4506 non-null   object
 5   news_url2       42 non-null     object
 6   news_url3       25 non-null     object
 7   news_url4       15 non-null     object
 8   news_url5       4 non-null      object
 9   title           4532 non-null   object
 10  newstitle       4069 non-null   object
 11  content         3706 non-null   object
 12  abstract        2473 non-null   object
 13  publish_date    1111 non-null   object
 14  meta_keywords   4069 non-null   object
 15  label           4532 non-null   int64 
dtypes: int64(2), object(14)
memory usage: 601.9+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 152288 en

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
news_ids = set(all_news["news_id"])
print("Total unique news:", len(news_ids))

Total unique news: 4532


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

bert_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [ ]:
@torch.no_grad()
def get_embedding(text):
    if pd.isna(text) or text.strip() == "":
        text = "empty"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    outputs = bert_model(**inputs)

    emb = outputs.last_hidden_state[:, 0, :]
    return emb.squeeze(0)[:128]   # reduce dimension

In [ ]:
import torch
import numpy as np
from torch_geometric.data import Data
from collections import defaultdict

def build_graph_sequence(news_id, num_snapshots=5):
    t = all_tweets[all_tweets["news_id"] == news_id]
    r = all_replies[all_replies["news_id"] == news_id]

    if len(t) == 0:
        return None

    # ✅ Get news embedding
    news_text = all_news[all_news["news_id"] == news_id]["content"].values
    if len(news_text) == 0:
        return None

    news_embedding = get_cached_embedding(news_text[0])

    # ✅ Pseudo-time (order-based)
    t = t.reset_index(drop=True)

    num_snapshots = min(num_snapshots, len(t))
    tweet_chunks = np.array_split(t, num_snapshots)

    snapshots = []

    for chunk in tweet_chunks:
        node_features = {}
        edges = []

        # -----------------------
        # Step 1: Build nodes + edges
        # -----------------------
        tweet_ids = set()

        for _, row in chunk.iterrows():
            tid = row["tweet_id"]
            tweet_ids.add(tid)
            node_features[tid] = None  # placeholder

        reply_subset = r[r["tweet_id"].isin(tweet_ids)]

        for _, row in reply_subset.iterrows():
            rid = row["reply_id"]
            pid = row["tweet_id"]

            node_features[rid] = None  # placeholder
            edges.append((pid, rid))

        if len(node_features) == 0:
            continue

        # -----------------------
        # Step 2: Compute DEGREE
        # -----------------------
        degree = defaultdict(int)

        for u, v in edges:
            degree[u] += 1
            degree[v] += 1

        # -----------------------
        # Step 3: Assign features
        # -----------------------
        for nid in node_features.keys():
            base = news_embedding.clone()

            deg = degree[nid]
            deg_norm = deg / (len(node_features) + 1)

            # role: tweet = 0, reply = 1
            is_reply = 1.0 if nid not in tweet_ids else 0.0

            # 🔥 final feature (same dim as embedding)
            feature = base + (deg_norm * 0.1) + (is_reply * 0.05)

            node_features[nid] = feature

        # -----------------------
        # Step 4: Build tensors
        # -----------------------
        node_list = list(node_features.keys())
        id_map = {nid: i for i, nid in enumerate(node_list)}

        x = torch.stack([node_features[nid] for nid in node_list])

        edge_index = []
        for u, v in edges:
            if u in id_map and v in id_map:
                edge_index.append([id_map[u], id_map[v]])

        if len(edge_index) == 0:
            edge_index = torch.zeros((2, 1), dtype=torch.long)
        else:
            edge_index = torch.tensor(edge_index).t().contiguous()

        data = Data(x=x, edge_index=edge_index)
        data.batch = torch.zeros(x.size(0), dtype=torch.long)

        snapshots.append(data)

    # 🔥 avoid Conv1D crash
    if len(snapshots) < 3:
        return None

    return snapshots

In [ ]:
class SEE(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=64):  # ✅ FIXED
        super().__init__()
        self.gcn = GCNConv(in_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim)

    def forward(self, snapshots):
        h = None
        outputs = []

        for data in snapshots:
            x = self.gcn(data.x, data.edge_index)
            x = F.relu(x)

            g = global_mean_pool(x, data.batch)
            g = g.unsqueeze(0)

            out, h = self.gru(g, h)
            outputs.append(out.squeeze(0))

        return torch.stack(outputs)

In [ ]:
class TIE(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_dim, hidden_dim, 2)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, 2)

    def forward(self, x):
        # x: [T, H]

        # 🔥 Correct condition
        if x.shape[0] < 3:
            return x.mean(dim=0)

        x = x.transpose(0, 1).unsqueeze(0)  # [1, H, T]

        x = F.relu(self.conv1(x))  # T -> T-1
        x = F.relu(self.conv2(x))  # T-1 -> T-2

        x = x.mean(dim=2)
        return x.squeeze(0)

In [ ]:
# class Model(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.conv1 = GCNConv(128, 64)
#         self.conv2 = GCNConv(64, 64)
#         self.dropout = nn.Dropout(0.5)
#         self.fc = nn.Linear(64, 2)

#     def forward(self, data):
#         x = F.relu(self.conv1(data.x, data.edge_index))
#         x = self.dropout(x)
#         x = F.relu(self.conv2(x, data.edge_index))
#         x = global_mean_pool(x, data.batch)
#         return self.fc(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class Model(nn.Module):
    def __init__(self, in_dim=128, hidden_dim=64, num_classes=2):
        super().__init__()

        # Spatial + Evolution Encoder
        self.see = SEE(in_dim, hidden_dim)

        # Temporal Interaction Encoder
        self.tie = TIE(hidden_dim)

        # Final classifier

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, snapshots):
        """
        snapshots: list of PyG Data objects (time steps)
        """

        # Step 1: SEE → captures spatial + temporal (GRU)
        x = self.see(snapshots)
        # shape: [T, batch_size, hidden_dim]

        # Step 2: Aggregate across batch if needed
        # (depends on your batching setup)
        x = x.mean(dim=1)
        # shape: [T, hidden_dim]

        # Step 3: TIE → temporal convolution
        x = self.tie(x)
        # shape: [hidden_dim]

        # Step 4: Classification
        out = self.fc(x)

        return out

In [ ]:
news_ids = list(all_news["news_id"])
labels = dict(zip(all_news["news_id"], all_news["label"]))

random.shuffle(news_ids)

split = int(0.8 * len(news_ids))
train_ids = news_ids[:split]

# oversample fake
fake_ids = [nid for nid in train_ids if labels[nid] == 1]
train_ids = train_ids + fake_ids * 5

# ✅ FIX: shuffle AFTER oversampling
random.shuffle(train_ids)

val_ids = news_ids[split:]

# safety limit
train_ids = train_ids[:200]
val_ids = val_ids[:50]

print("Fake count:", len([nid for nid in train_ids if labels[nid] == 1]))

Fake count: 91


In [ ]:
@torch.no_grad()
def evaluate_full(model, ids):
    model.eval()

    preds, true = [], []

    for nid in ids:
        snapshots = build_graph_sequence(nid)  # ✅ FIX

        if snapshots is None:
            continue

        logits = model(snapshots)

        pred = torch.argmax(logits).item()

        preds.append(pred)
        true.append(labels[nid])

    if len(true) == 0:
        return None

    print("Pred counts:", {i: preds.count(i) for i in set(preds)})
    print("True counts:", {i: true.count(i) for i in set(true)})

    acc = accuracy_score(true, preds)

    precision = precision_score(true, preds, zero_division=0)
    recall = recall_score(true, preds, zero_division=0)
    f1 = f1_score(true, preds, zero_division=0)
    f1_macro = f1_score(true, preds, average='macro', zero_division=0)
    cm = confusion_matrix(true, preds)

    return {
        "acc": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f1_macro": f1_macro,
        "cm": cm
    }

In [ ]:
print(all_tweets.columns)
print(all_replies.columns)

Index(['news_id', 'tweet_id'], dtype='object')
Index(['news_id', 'tweet_id', 'reply_id'], dtype='object')


In [ ]:
embedding_cache = {}

def get_cached_embedding(text):
    if text not in embedding_cache:
        embedding_cache[text] = get_embedding(text)
    return embedding_cache[text]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [ ]:
model = Model()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
from collections import Counter

# Count labels in training set
counts = Counter([labels[nid] for nid in train_ids])
print("Class counts:", counts)

total = sum(counts.values())

# Compute weights
weight_real = total / counts[0]
weight_fake = total / counts[1]

weights = torch.tensor([weight_real, weight_fake], dtype=torch.float)

print("Class weights:", weights)

# Use weighted loss
criterion = nn.CrossEntropyLoss(weight=weights)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    random.shuffle(train_ids)

    for nid in train_ids:
        snapshots = build_graph_sequence(nid)

        if snapshots is None:
            continue

        logits = model(snapshots)
        label = torch.tensor([labels[nid]])

        loss = criterion(logits.unsqueeze(0), label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ✅ Compute metrics
    train_metrics = evaluate_full(model, train_ids)
    val_metrics = evaluate_full(model, val_ids)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")

    if train_metrics:
        print("\nTRAIN:")
        print(f"Acc: {train_metrics['acc']:.4f}")
        print(f"Precision: {train_metrics['precision']:.4f}")
        print(f"Recall: {train_metrics['recall']:.4f}")
        print(f"F1: {train_metrics['f1']:.4f}")
        print(f"F1_macro: {train_metrics['f1_macro']:.4f}")

    if val_metrics:
        print("\nVALIDATION:")
        print(f"Acc: {val_metrics['acc']:.4f}")
        print(f"Precision: {val_metrics['precision']:.4f}")
        print(f"Recall: {val_metrics['recall']:.4f}")
        print(f"F1: {val_metrics['f1']:.4f}")
        print(f"F1_macro: {val_metrics['f1_macro']:.4f}")

        print("\nConfusion Matrix:")
        print(val_metrics["cm"])

    print("="*60)

Class counts: Counter({0: 109, 1: 91})
Class weights: tensor([1.8349, 2.1978])


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Pred counts: {0: 60, 1: 61}
True counts: {0: 50, 1: 71}


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Pred counts: {0: 17, 1: 5}
True counts: {0: 16, 1: 6}

Epoch 1
Loss: 78.8028

TRAIN:
Acc: 0.8182
Precision: 0.9016
Recall: 0.7746
F1: 0.8333
F1_macro: 0.8167

VALIDATION:
Acc: 0.8636
Precision: 0.8000
Recall: 0.6667
F1: 0.7273
F1_macro: 0.8182

Confusion Matrix:
[[15  1]
 [ 2  4]]


...

Pred counts: {0: 17, 1: 5}
True counts: {0: 16, 1: 6}

Epoch 4
Loss: 50.3420

TRAIN:
Acc: 0.9091
Precision: 0.9167
Recall: 0.9296
F1: 0.9231
F1_macro: 0.9060

VALIDATION:
Acc: 0.8636
Precision: 0.8000
Recall: 0.6667
F1: 0.7273
F1_macro: 0.8182

Confusion Matrix:
[[15  1]
 [ 2  4]]


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Pred counts: {0: 44, 1: 77}
True counts: {0: 50, 1: 71}


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Pred counts: {0: 16, 1: 6}
True counts: {0: 16, 1: 6}

Epoch 5
Loss: 69.7462

TRAIN:
Acc: 0.9339
Precision: 0.9091
Recall: 0.9859
F1: 0.9459
F1_macro: 0.9304

VALIDATION:
Acc: 0.8182
Precision: 0.6667
Recall: 0.6667
F1: 0.6667
F1_macro: 0.7708

Confusion Matrix:
[[14  2]
 [ 2  4]]
